#### Step 1: Read Plate Information

In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
from matplotlib.ticker import LogFormatter
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import json
import plategig 

PROJECT_PATH = Path('data/external/projects')
PROJECT_ID = '241011_Adam'

In [ ]:
# Read the Excel file into a DataFrame
df_plate_info = pd.read_excel(PROJECT_PATH / PROJECT_ID / 'data' / 'PlateInfo.xlsx', engine='openpyxl')
df_plate_info

In [ ]:
def calculate_plate_id(row):
    if row['Experiment'] == 1:
        return row['Plate']
    elif row['Experiment'] == 2:
        return row['Plate'] + 99
    elif row['Experiment'] == 3:
        return row['Plate'] + 148
    else:
        return None  # Handle other experiments if needed

# Step 2: Apply the function to each row in the DataFrame and create the new 'Plate_ID' column
df_plate_info['Plate_ID'] = df_plate_info.apply(calculate_plate_id, axis=1)

# Step 3: Display the updated DataFrame
print(df_plate_info.head())

Collect unique plate IDs

unique_plate_ids, plate_numbers = growthlib.get_unique_plate_ids(df_plate_info)
unique_plate_ids

#### Step 2: Read the raw OD data

In [ ]:
df_OD_raw = pd.read_excel(PROJECT_PATH / PROJECT_ID / 'data' /'ODFinal.xlsx')
# drop last nan column, fix to input mistake
df_OD_raw = df_OD_raw.iloc[:,:-1]
df_OD_raw.tail()

In [ ]:
df = plategig.static.convert_OD_plate_to_long(df_OD_raw, df_plate_info)
df

Calculate median background of all plates to impute plates with missing media-only wells.

In [ ]:
df_plate_info[df_plate_info['Strain']=='Media Only']

In [ ]:

median_background_all_plates = plategig.static.calc_median_background_all_plates(
    df, df_plate_info, plot=True)
# Normalize plate-map identifiers before joining measurements.
print(median_background_all_plates)

In [ ]:
plategig.static.plot_single_plate_media_only_wells(df, df_plate_info, plate_id=1, column_name='OD')

Apply background correction for all plates

In [ ]:
df_bc = df.copy()
df_bc['OD_final'] = df_bc['OD'] - median_background_all_plates

#### Evaluate growth metrics

In [ ]:
df_bc

In [ ]:
# Prep the analysis dataframe
df_analysis = pd.merge(df_plate_info, df_bc, on=['Plate_ID', 'Well'])
df_analysis = df_analysis[['Experiment','Strain', 'Culture', 'Replicate', 'Antibiotic',
                           'Dose', 'Plate_ID', 'Well', 'Row', 'Column',
                           'OD', 'OD_final']]
# Remove control wells
df_analysis = df_analysis[~df_analysis['Strain'].isin(['Media Only','Cells Only'])]
# Keep group names
df_analysis['Group'] = df_analysis.Strain
# Define each culture of as a separate strain
df_analysis['Strain'] = df_analysis.Strain + df_analysis.Culture.astype(int).astype(str)
df_analysis


In [ ]:
# Exclusion criteria:
# Exclude failed measurements:
# From experiment 1
#   ‘strain’ == ‘PAC’         (not used)
#   ‘strain’ == PCr             (range to small)
#   cultures == PLAC7 & PLAC10        (bad curve)
# From Experiment 2
#    ATEC 1,2,4     (plating error)
#    ATEC-C 3       (plating error)
#    ATEC-C-r        (plating error)
### Groups to remove
    # PAC
    # P
# Define the combinations of 'Experiment' and 'Group' to remove
df_analysis['Experiment'] = df_analysis['Experiment'].astype(str).str.strip()

groups_to_remove = [('1', 'PAC'), ('1', 'PCr'), ('2', 'ATEC-C-R')]

# Remove rows where the 'Experiment' and 'Group' match the combinations in groups_to_remove
df_analysis = df_analysis[~df_analysis[['Experiment', 'Group']].apply(tuple, axis=1).isin(groups_to_remove)]

# Define the combinations of 'Experiment' and 'Strain' to remove
strains_to_remove = [('1', 'PLAC7'), ('1', 'PLAC10'), ('2', 'ATEC1'), ('2', 'ATEC2'), ('2', 'ATEC4'), ('2', 'ATEC-C3')]

# Remove rows where the 'Experiment' and 'Strain' match the combinations in strains_to_remove
df_analysis = df_analysis[~df_analysis[['Experiment', 'Strain']].apply(tuple, axis=1).isin(strains_to_remove)]


In [ ]:
# Select subgroups
df_analysis = df_analysis.query("Group.isin(['P','PA','PC','PL'])")
df_analysis

In [ ]:
### Double check
unique_columns = df_analysis.columns.tolist()

# Print the list of unique columns
print(unique_columns)

#
unique_values = df_analysis['Strain'].unique()

# Print the unique values
print(unique_values)

In [ ]:
antibiotics = df_analysis['Antibiotic'].unique()
antibiotics 

In [ ]:
df_analysis.query("Antibiotic == 'Levofloxacin' & Strain == 'P1'")

In [ ]:

# Adjust the figure size as needed
fig, ax = plt.subplots(figsize=(3, 2))
plategig.static.plot_dose_response_curve_errorbar(df_analysis, 'P2', 'Levofloxacin', strain_colors={'P1':'black'}, ax=ax)

In [ ]:
# Plot all dose-response curves in the df_analysis tables (might take too long)
for drug in antibiotics:
    if not pd.isna(drug):
        plategig.static.plot_od_final_for_selected_antibiotic(
            df_analysis,
            plategig.static.plot_dose_response_curve_errorbar,
            drug,
            strain_colors={})

In [ ]:
valid_combinations = plategig.static.prep_valid_combinations(
    df_analysis,
    multiplex=['Strain', 'Antibiotic'],
    ic50_threshold=0.5,
    mic_threshold=0.05)
valid_combinations

In [ ]:
growth_features = plategig.static.apply_phenotyper(df_analysis, valid_combinations)
growth_features = plategig.static.cap_growth_features_within_experiment_range(growth_features)
growth_features

#growth_features.to_excel('data/external/growth_features.csv', index=False)



In [ ]:
import re
growth_features['group'] = growth_features['Strain'].apply(lambda x: re.split(r'[1-9]', x)[0])
growth_features

In [ ]:
growth_features.to_excel('data/external/growth_features.xlsx', index=False)

In [ ]:
from matplotlib.ticker import FuncFormatter
plt.rcParams["font.family"] = "Nimbus Roman"

fig, ax = plt.subplots(figsize=(3, 2.5), dpi=150)
plategig.static.plot_dose_response_curve_fit(df_analysis,
                                              strain='PA5',
                                              antibiotic='Levofloxacin',
                                              growth_features=growth_features,
                                              strain_colors={},
                                              ax=ax)

# Define a custom formatter for regular (non-scientific) labels
def log_formatter(value, tick_position):
    rounded_value = round(value, 3)  # Round to the nearest hundredth
    return f"{rounded_value:.3f}".rstrip('0').rstrip('.')  # Always display with two decimal places

# Apply the custom formatter to the x-axis
ax.xaxis.set_major_formatter(FuncFormatter(log_formatter))

# Get current tick labels and set every other one to be blank
for i, label in enumerate(ax.get_xticklabels()):
    if i % 2 != 0:  # For every odd index, make it invisible
        label.set_visible(False)

# Adjust font size
ax.tick_params(axis='x', labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Customize axis labels
ax.set_xlabel("Levofloxacin (µg/mL)", fontsize=14)
ax.set_ylabel("Absorbance (OD$_{600}$)", fontsize=14) 
fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures' / 'finals'/'0512_finals'/ 'PA5-levofloxacin.png', dpi=600, bbox_inches='tight')

In [ ]:
strains = list(set(df_analysis['Strain'].str.lower()) - set(['media only', 'cells only', np.nan]))
print(strains)

In [ ]:
antibiotics = list(set(df_analysis['Antibiotic'].str.lower()) - set(['media only', 'cells only', np.nan]))
antibiotics

In [ ]:
strain_colors = {strain:color for strain, color in zip(strains, sns.color_palette('muted')[:len(strains)])}


In [ ]:
num_drugs = growth_features['Antibiotic'].nunique()
num_experiments = growth_features.shape[0]
num_cols = 6
num_rows = num_experiments//num_cols + (num_experiments % num_cols > 0)
# num_cols = num_experiments//num_drugs
# fig, axes = plt.subplots(num_experiments//num_replicates, num_replicates,
#                        figsize=(num_replicates*3, num_experiments*2), dpi=90)
fig, axes = plt.subplots(num_rows, num_cols,
                       figsize=(num_cols*3, num_rows*2), dpi=300)

for ix, row in growth_features.iterrows():
    plategig.static.plot_dose_response_curve_fit(df_analysis,
                                                growth_features=growth_features,
                                                strain_colors=strain_colors,
                                                strain=row['Strain'],
                                                antibiotic=row['Antibiotic'],
                                                ax=axes.flat[ix])
    
    if row['Status'] == 'FAIL':
        axes.flat[ix].text(0.5, 0.5, f'FAIL', color='red', fontsize=12, fontweight='bold',
                        ha='center', va='center', transform=axes.flat[ix].transAxes)
        # gray out the entire axes
        axes.flat[ix].set_facecolor('lightgray')
    if ix % num_cols != 0:
        axes.flat[ix].set_yticklabels([])
# Hide the rest of the axes
for ix in range(num_experiments, num_cols*num_rows):
    axes.flat[ix].axis('off')
    
fig.tight_layout()
# set empty space between subplots
fig.subplots_adjust(wspace=0.05)
#fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures' / f'IC50 fits.png', dpi=300, bbox_inches='tight')

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"

manual_colors = {
    "P": "grey",
    "PA": "blue",
    "PC": "forestgreen",
    "PL": "red"
}

def cohort_label(group):
    if group == "P":
        return "MG"
    elif group == "PA":
        return "MG$^{\\mathrm{AMI}}$"
    elif group == "PC":
        return "MG$^{\\mathrm{CEF}}$"
    elif group == "PL":
        return "MG$^{\\mathrm{LEV}}$"
    else:
        return group  # Default to the original group name if no match
    
def custom_label(strain):
    if strain.startswith("P") and strain[1:].isdigit():
        # General case: P1 through P10
        number = strain[1:]  # Extract number
        return number
    elif strain.startswith("PA") and strain[2:].isdigit():
        # PA1 through PA10
        number = strain[2:]  # Extract number
        return f"MG$^{{\\mathrm{{AMI}}}}$-{number}"
    elif strain.startswith("PC") and strain[2:].isdigit():
        # PC1 through PC10
        number = strain[2:]  # Extract number
        return f"MG$^{{\\mathrm{{CEF}}}}$-{number}"
    elif strain.startswith("PL") and strain[2:].isdigit():
        # PL1 through PL10
        number = strain[2:]  # Extract number
        return f"MG$^{{\\mathrm{{LEV}}}}$-{number}"
    else:
        # Default to the original strain label if no match
        return strain

# Define desired order for StrainGroup
desired_order = ["P", "PA", "PL", "PC"]

# Modify StrainGroup to reflect the desired order
growth_features['StrainGroup'] = pd.Categorical(
    growth_features['Strain'].str.extract(r'(\D+)')[0],
    categories=desired_order,
    ordered=True
)

# Add a column for Culture Number
growth_features['CultureNumber'] = growth_features['Strain'].str.extract(r'(\d+)$').astype(int)

# Sort by StrainGroup and CultureNumber
growth_features = growth_features.sort_values(['StrainGroup', 'CultureNumber'])
print("Order of strains to be plotted:")
print(growth_features['Strain'].to_list())
# Dictionary of custom y-axis ranges for specific drugs
y_axis_ranges = {
    "Levofloxacin": (0.001, 10),
    "Amikacin": (.1, 1000),
    "Cefepime": (0.001, 10)
}

# Iterate over antibiotics to create subplots
fig, axes = plt.subplots(
    nrows=len(sorted(growth_features['Antibiotic'].unique())),
    figsize=(14, 18),
    constrained_layout=True
)

# Ensure axes is iterable even for a single plot
if len(sorted(growth_features['Antibiotic'].unique())) == 1:
    axes = [axes]

for ax, drug in zip(axes, sorted(growth_features['Antibiotic'].unique())):
    # Filter data for the current drug
    filtered_data = growth_features.query(f'Antibiotic == "{drug}" & Status == "PASS"')
    filtered_data['StrainGroup'] = filtered_data['Strain'].str.extract(r'(\D+)')

    # Apply custom labels
    filtered_data['CustomLabel'] = filtered_data['Strain'].apply(custom_label)

    # Extract values for plotting
    strains = filtered_data['CustomLabel']
    ic50 = filtered_data['IC50']
    ci_lower = filtered_data['IC50_ci_lower']
    ci_upper = filtered_data['IC50_ci_upper']
    insufficient_drug = filtered_data['insufficient_drug']
    strain_groups = filtered_data['StrainGroup']

    # Assign colors for each strain group
    unique_groups = strain_groups.unique()
    group_colors = {group: manual_colors.get(group, "black") for group in unique_groups}
    colors = strain_groups.map(group_colors)

    # Plot IC50 points
    scatter = ax.scatter(strains, ic50, color=colors, s=100, label=strain_groups)
    
    # Add error bars with matching colors
    for i, color in enumerate(colors):
        ax.errorbar(
            x=i,
            y=ic50.iloc[i],
            yerr=[[ic50.iloc[i] - ci_lower.iloc[i]], [ci_upper.iloc[i] - ic50.iloc[i]]],
            fmt='none',
            ecolor=color,
            capsize=3
        )

    # Add dagger annotations for insufficient drug
    for i, strain in enumerate(strains):
        if insufficient_drug.iloc[i]:
            ax.text(i, ic50.iloc[i] * 1.2, '‡', ha='center', va='bottom', fontsize=12, color='black')

    # Format the y-axis with log scale
    ax.set_yscale('log')
    
    # Apply custom y-axis range if the drug is in the dictionary
    if drug in y_axis_ranges:
        ax.set_ylim(y_axis_ranges[drug])

    ax.set_ylabel(f"{drug}\nIC$_{{50}}$ (μg/mL)", fontsize=22)
    ax.tick_params(axis='y', labelsize=22)  # Adjust 'labelsize' to your preferred size

    ax.set_xlim(-0.5, len(strains) - 0.5)  # Tighten edges by adding a small buffer

    for spine in ax.spines.values():
        spine.set_linewidth(2)  # Set thickness of the spine lines


    ax.tick_params(axis='both', which='major', length=12, width=3)  # Major ticks
    ax.tick_params(axis='both', which='minor', length=8, width=2)  # Minor ticks

    if drug == 'Levofloxacin':
        ax.set_xlabel('Culture', fontsize=22, labelpad=40)         
        # Calculate repeated labels
        repeated_labels = [str(i) for i in range(1, 11)] * (len(strains) // 10 + 1)
        repeated_labels = repeated_labels[:len(strains)]
        
        # Set xticks and xticklabels
        ax.set_xticks(range(len(strains)))
        ax.set_xticklabels(repeated_labels, fontsize=15)

        # Calculate unique cohort groups and their midpoints
        cohort_midpoints = []
        current_group = strain_groups.iloc[0]
        group_start = 0

        for i, group in enumerate(strain_groups):
            if group != current_group:
                midpoint = (group_start + i - 1) / 2
                cohort_midpoints.append((current_group, midpoint))
                group_start = i
                current_group = group
        
        # Add the last group
        midpoint = (group_start + len(strain_groups) - 1) / 2
        cohort_midpoints.append((current_group, midpoint))

        # Add cohort labels below the x-axis
        for group, midpoint in cohort_midpoints:
            ax.text(midpoint / len(strains)*1.02, -0.1, cohort_label(group), fontsize=18, 
                    ha='center', va='top', transform=ax.transAxes)

        # Create a legend with cohort labels
        handles = [plt.Line2D([0], [0], marker='o', color=color, markersize=10, linestyle='', 
                              label=cohort_label(group)) for group, color in group_colors.items()]
        ax.legend(handles=handles, title="Cohort", loc="upper left", fontsize=15, title_fontsize=15)

    else:
        ax.set_xticklabels("")  # Keep empty xticklabels for other subplots
        
final_chart_path = PROJECT_PATH / PROJECT_ID / 'figures'/'finals'/'0512_finals' / 'PLPAPC_IC50_individual_1205.png'
plt.savefig(final_chart_path, dpi=600, bbox_inches='tight')
plt.show()
# Show plot
plt.show()
filtered_data


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"
manual_colors = {
    "P": "grey",
    "PA": "blue",
    "PC": "forestgreen",
    "PL": "red"
}
# Function for cohort labels
def cohort_label(group):
    if group == "P":
        return "MG"
    elif group == "PA":
        return "MG$^{\\mathrm{AMI}}$"
    elif group == "PC":
        return "MG$^{\\mathrm{CEF}}$"
    elif group == "PL":
        return "MG$^{\\mathrm{LEV}}$"
    else:
        return group  # Default to the original group name if no match

# Function for custom strain labels
def custom_label(strain):
    if strain.startswith("P") and strain[1:].isdigit():
        number = strain[1:]  # Extract number
        return f"MG-{number}"
    elif strain.startswith("PA") and strain[2:].isdigit():
        number = strain[2:]  # Extract number
        return f"MG$^{{\\mathrm{{AMI}}}}$-{number}"
    elif strain.startswith("PC") and strain[2:].isdigit():
        number = strain[2:]  # Extract number
        return f"MG$^{{\\mathrm{{CEF}}}}$-{number}"
    elif strain.startswith("PL") and strain[2:].isdigit():
        number = strain[2:]  # Extract number
        return f"MG$^{{\\mathrm{{LEV}}}}$-{number}"
    else:
        return strain
# Define the desired order of Strain groups
# Define desired order for StrainGroup
desired_order = ["P", "PA", "PL", "PC"]

# Modify StrainGroup to reflect the desired order
growth_features['StrainGroup'] = pd.Categorical(
    growth_features['Strain'].str.extract(r'(\D+)')[0],
    categories=desired_order,
    ordered=True
)

# Add a column for Culture Number
growth_features['CultureNumber'] = growth_features['Strain'].str.extract(r'(\d+)$').astype(int)

# Sort by StrainGroup and CultureNumber
growth_features = growth_features.sort_values(['StrainGroup', 'CultureNumber'])
print("Order of strains to be plotted:")
print(growth_features['Strain'].to_list())
# Sort the dataframe based on the new StrainGroup order

# Custom y-axis ranges for specific drugs
y_axis_ranges = {
    "Levofloxacin": (0.01, 100),
    "Amikacin": (1, 10000),
    "Cefepime": (0.01, 100)
}

# Create subplots
fig, axes = plt.subplots(
    nrows=len(sorted(growth_features['Antibiotic'].unique())),
    figsize=(14, 18),
    constrained_layout=True
)

# Ensure axes is iterable for a single plot
if len(sorted(growth_features['Antibiotic'].unique())) == 1:
    axes = [axes]

for ax, drug in zip(axes, sorted(growth_features['Antibiotic'].unique())):
    # Filter data for the current drug
    filtered_data = growth_features.query(f'Antibiotic == "{drug}" & Status == "PASS"')
    filtered_data['StrainGroup'] = filtered_data['Strain'].str.extract(r'(\D+)')
    filtered_data['CustomLabel'] = filtered_data['Strain'].apply(custom_label)

    # Extract values for plotting
    strains = filtered_data['CustomLabel']
    mic = filtered_data['MIC']
    mic_lower = filtered_data['MIC_ci_lower']
    mic_upper = filtered_data['MIC_ci_upper']
    insufficient_drug = filtered_data['insufficient_drug']
    strain_groups = filtered_data['StrainGroup']

    # Assign colors for each cohort
    unique_groups = strain_groups.unique()
    group_colors = {group: manual_colors.get(group, "black") for group in unique_groups}
    colors = strain_groups.map(group_colors)

    # Plot MIC points
    scatter = ax.scatter(strains, mic, color=colors, s=100, label=strain_groups)
    
    # Add error bars with matching colors
    for i, color in enumerate(colors):
        ax.errorbar(
            x=i,
            y=mic.iloc[i],
            yerr=[[mic.iloc[i] - mic_lower.iloc[i]], [mic_upper.iloc[i] - mic.iloc[i]]],
            fmt='none',
            ecolor=color,
            capsize=3
        )

    # Add dagger annotations for insufficient drug
    for i, strain in enumerate(strains):
        if insufficient_drug.iloc[i]:
            ax.text(i, mic.iloc[i] * 1.2, '‡', ha='center', va='bottom', fontsize=12, color='black')

    # Format the y-axis with log scale
    ax.set_yscale('log')
    
    # Apply custom y-axis range if the drug is in the dictionary
    if drug in y_axis_ranges:
        ax.set_ylim(y_axis_ranges[drug])

    ax.set_ylabel(f"{drug}\nMIC (μg/mL)", fontsize=22)
    ax.tick_params(axis='y', labelsize=22)  # Adjust 'labelsize' to your preferred size

    ax.set_xlim(-0.5, len(strains) - 0.5)  # Tighten edges by adding a small buffer

    for spine in ax.spines.values():
        spine.set_linewidth(2)  # Set thickness of the spine lines


    ax.tick_params(axis='both', which='major', length=12, width=3)  # Major ticks
    ax.tick_params(axis='both', which='minor', length=8, width=2)  # Minor ticks

    if drug == 'Levofloxacin':
        ax.set_xlabel('Culture', fontsize=22, labelpad=40)         
        # Calculate repeated labels
        repeated_labels = [str(i) for i in range(1, 11)] * (len(strains) // 10 + 1)
        repeated_labels = repeated_labels[:len(strains)]
        
        # Set xticks and xticklabels
        ax.set_xticks(range(len(strains)))
        ax.set_xticklabels(repeated_labels, fontsize=15)

        # Calculate unique cohort groups and their midpoints
        cohort_midpoints = []
        current_group = strain_groups.iloc[0]
        group_start = 0

        for i, group in enumerate(strain_groups):
            if group != current_group:
                midpoint = (group_start + i - 1) / 2
                cohort_midpoints.append((current_group, midpoint))
                group_start = i
                current_group = group
        
        # Add the last group
        midpoint = (group_start + len(strain_groups) - 1) / 2
        cohort_midpoints.append((current_group, midpoint))

        # Add cohort labels below the x-axis
        for group, midpoint in cohort_midpoints:
            ax.text(midpoint / len(strains)*1.02, -0.1, cohort_label(group), fontsize=18, 
                    ha='center', va='top', transform=ax.transAxes)

        # Create a legend with cohort labels
        handles = [plt.Line2D([0], [0], marker='o', color=color, markersize=10, linestyle='', 
                              label=cohort_label(group)) for group, color in group_colors.items()]
        ax.legend(handles=handles, title="Cohort", loc="upper left", fontsize=15, title_fontsize=15)

    else:
        ax.set_xticklabels("")  # Keep empty xticklabels for other subplots


final_chart_path = PROJECT_PATH / PROJECT_ID / 'figures' / 'finals'/'0512_finals' / 'PLPAPC_MIC_individual_1205.png'
plt.savefig(final_chart_path, dpi=600, bbox_inches='tight')


plt.show()


In [ ]:
(
growth_features[['Antibiotic','Strain','Status','IC50','MIC','IC50_ci_lower','IC50_ci_upper', 'max_growth', 'hill_coeff']]
.to_csv(PROJECT_PATH / PROJECT_ID / 'figures' / f'IC50 estimations.csv',
          index=False)
)

In [ ]:
import seaborn as sns

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"


########## NO LABELS

# Assign the antibiotic you want to graph
selected_antibiotic = "Levofloxacin"

# Subset the data for the selected antibiotic
subset_data = growth_features[growth_features['Antibiotic'] == selected_antibiotic].copy()  # Avoid SettingWithCopyWarning

# Custom y-axis limits for each antibiotic
custom_y_limits = {
    "Levofloxacin": (0.001, 10),
    "Amikacin": (.1, 1000),
    "Cefepime": (0.001, 10)
}


# Create a new column for the updated group labels
def assign_group_label(row):
    if row['group'] == "P":
        return r"MG"  # Just MG
    elif row['group'] == "PA":
        return r"MG$^{\mathrm{AMI}}$"  # MG with AMI as superscript
    elif row['group'] == "PC":
        return r"MG$^{\mathrm{CEF}}$"  # MG with CEF as superscript
    elif row['group'] == "PL":
        return r"MG$^{\mathrm{LEV}}$"  # MG with LEV as superscript
    else:
        return "Unknown"

# Ensure the 'group' column exists
if 'group' not in subset_data.columns:
    subset_data['group'] = growth_features['Strain'].apply(lambda x: re.split(r'[1-9]', x)[0])

# Apply the function to create the new column
subset_data.loc[:, 'group_fancy'] = subset_data.apply(assign_group_label, axis=1)

# Update the palette to match the new group labels
group_palette = {
    r"MG": "grey",
    r"MG$^{\mathrm{AMI}}$": "blue",
    r"MG$^{\mathrm{CEF}}$": "forestgreen",
    r"MG$^{\mathrm{LEV}}$": "red",
}
order = [r"MG", r"MG$^{\mathrm{AMI}}$", r"MG$^{\mathrm{LEV}}$", r"MG$^{\mathrm{CEF}}$"]

# Use FacetGrid for plotting
g = sns.FacetGrid(
    subset_data,
    col="Antibiotic",  # Facet by Antibiotic
    sharey=False,  # Allow independent y-axis scaling
    height=10,  # Height of each facet
    aspect=1.5,  # Aspect ratio for width scaling
)

# Map bar plots to the grid
g.map(
    sns.barplot,
    "group_fancy",  # X-axis
    "IC50",  # Y-axis
    ci="sd",
    order=order,  # Ensure consistent x-axis order
    palette=group_palette,
    alpha=0.6,
    errcolor="black",  # Color of the error bars
    errwidth=3,  # Thickness of the error bars
    capsize=0.05  # Length of the caps on error bars
)

# Custom function to calculate evenly spaced x-positions
def get_evenly_spaced_positions(group_data, x_base, spread=0.36, gap=0.08):
    """
    Calculate evenly spaced x-positions around a central x_base, leaving a gap in the middle.

    Parameters:
        group_data: DataFrame containing points for a single category.
        x_base: Central x position for the category.
        spread: Maximum range to spread points around the center.
        gap: The size of the gap to leave in the middle (default: 0.1).

    Returns:
        List of adjusted x-positions for the points.
    """
    n_points = len(group_data)
    if n_points == 1:
        return [x_base]  # No need to spread if there's only one point

    half_points = n_points // 2  # Divide points into two halves
    if n_points % 2 == 0:
        # Even number of points
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        offsets = np.concatenate([left_offsets, right_offsets])
    else:
        # Odd number of points: one point stays at the center
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        center_offset = [0]  # Keep one point at the exact center
        offsets = np.concatenate([left_offsets, center_offset, right_offsets])

    return x_base + offsets

# Overlay strip plots with custom spacing
for group_fancy, group_data in subset_data.groupby('group_fancy'):
    x_base = order.index(group_fancy)  # Base x position based on the custom order
    x_positions = get_evenly_spaced_positions(group_data, x_base)
    
    g.ax.scatter(
        x_positions,
        group_data['IC50'],
        color="black",
        facecolor="black",
        s=100,
        edgecolor="black",
        linewidth=2,
        zorder=10  # Ensure points are above other plot elements
    )

# Customizations for the y-axis and ticks
for ax in g.axes.flat:
    ax.set_yscale("log")  # Set log scale

    # Apply custom y-axis limits for the selected antibiotic first
    if selected_antibiotic in custom_y_limits:
        ax.set_ylim(custom_y_limits[selected_antibiotic])

    # Now adjust font size for all y-tick labels (after y-limits are set)
    for label in ax.get_yticklabels():
        label.set_fontsize(37)

    ax.set_xticks([])  # Remove tick positions
    ax.set_xticklabels([])  # Remove tick labels

    ax.set_xlabel('')  # Remove x-axis label
    for spine in ax.spines.values():
        spine.set_linewidth(4)

    ax.tick_params(axis='both', which='major', length=12, width=3)
    ax.tick_params(axis='both', which='minor', length=8, width=2)

# Add axis labels and facet titles
g.set_axis_labels("", "", fontsize=40)
g.set_axis_labels("", f"{selected_antibiotic}\nIC$_{{50}}$ (μg/mL)", fontsize=40)
g.set_titles("", size=50)  # Facet title uses the column value

# Adjust overall layout and spacing
g.fig.subplots_adjust(left=0.2, right=0.9, top=0.9, wspace=0.3, hspace=0.4)
g.fig.set_size_inches(15, 10)  # Total figure size

# Save the plot (optional)

g.fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures'/'finals'/'0512_finals'/ f"{selected_antibiotic}_PLPAPC_IC50.png", dpi=600, bbox_inches="tight")

# Show the plot
plt.show()


In [ ]:
import seaborn as sns

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"


########## NO LABELS

# Assign the antibiotic you want to graph
selected_antibiotic = "Amikacin"

# Subset the data for the selected antibiotic
subset_data = growth_features[growth_features['Antibiotic'] == selected_antibiotic].copy()  # Avoid SettingWithCopyWarning

# Custom y-axis limits for each antibiotic
custom_y_limits = {
    "Levofloxacin": (0.001, 10),
    "Amikacin": (.1, 1000),
    "Cefepime": (0.001, 10)
}

# Create a new column for the updated group labels
def assign_group_label(row):
    if row['group'] == "P":
        return r"MG"  # Just MG
    elif row['group'] == "PA":
        return r"MG$^{\mathrm{AMI}}$"  # MG with AMI as superscript
    elif row['group'] == "PC":
        return r"MG$^{\mathrm{CEF}}$"  # MG with CEF as superscript
    elif row['group'] == "PL":
        return r"MG$^{\mathrm{LEV}}$"  # MG with LEV as superscript
    else:
        return "Unknown"

# Ensure the 'group' column exists
if 'group' not in subset_data.columns:
    subset_data['group'] = growth_features['Strain'].apply(lambda x: re.split(r'[1-9]', x)[0])

# Apply the function to create the new column
subset_data.loc[:, 'group_fancy'] = subset_data.apply(assign_group_label, axis=1)

# Update the palette to match the new group labels
group_palette = {
    r"MG": "grey",
    r"MG$^{\mathrm{AMI}}$": "blue",
    r"MG$^{\mathrm{CEF}}$": "forestgreen",
    r"MG$^{\mathrm{LEV}}$": "red",
}
order = [r"MG", r"MG$^{\mathrm{AMI}}$", r"MG$^{\mathrm{LEV}}$", r"MG$^{\mathrm{CEF}}$"]

# Use FacetGrid for plotting
g = sns.FacetGrid(
    subset_data,
    col="Antibiotic",  # Facet by Antibiotic
    sharey=False,  # Allow independent y-axis scaling
    height=10,  # Height of each facet
    aspect=1.5,  # Aspect ratio for width scaling
)

# Map bar plots to the grid
g.map(
    sns.barplot,
    "group_fancy",  # X-axis
    "IC50",  # Y-axis
    ci="sd",
    order=order,  # Ensure consistent x-axis order
    palette=group_palette,
    alpha=0.6,
    errcolor="black",  # Color of the error bars
    errwidth=3,  # Thickness of the error bars
    capsize=0.05  # Length of the caps on error bars
)

# Custom function to calculate evenly spaced x-positions
def get_evenly_spaced_positions(group_data, x_base, spread=0.36, gap=0.08):
    """
    Calculate evenly spaced x-positions around a central x_base, leaving a gap in the middle.

    Parameters:
        group_data: DataFrame containing points for a single category.
        x_base: Central x position for the category.
        spread: Maximum range to spread points around the center.
        gap: The size of the gap to leave in the middle (default: 0.1).

    Returns:
        List of adjusted x-positions for the points.
    """
    n_points = len(group_data)
    if n_points == 1:
        return [x_base]  # No need to spread if there's only one point

    half_points = n_points // 2  # Divide points into two halves
    if n_points % 2 == 0:
        # Even number of points
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        offsets = np.concatenate([left_offsets, right_offsets])
    else:
        # Odd number of points: one point stays at the center
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        center_offset = [0]  # Keep one point at the exact center
        offsets = np.concatenate([left_offsets, center_offset, right_offsets])

    return x_base + offsets

# Overlay strip plots with custom spacing
for group_fancy, group_data in subset_data.groupby('group_fancy'):
    x_base = order.index(group_fancy)  # Base x position based on the custom order
    x_positions = get_evenly_spaced_positions(group_data, x_base)
    
    g.ax.scatter(
        x_positions,
        group_data['IC50'],
        color="black",
        facecolor="black",
        s=100,
        edgecolor="black",
        linewidth=2,
        zorder=10  # Ensure points are above other plot elements
    )

# Customizations for the y-axis and ticks
for ax in g.axes.flat:
    ax.set_yscale("log")  # Set log scale

    # Apply custom y-axis limits for the selected antibiotic first
    if selected_antibiotic in custom_y_limits:
        ax.set_ylim(custom_y_limits[selected_antibiotic])

    # Now adjust font size for all y-tick labels (after y-limits are set)
    for label in ax.get_yticklabels():
        label.set_fontsize(37)

    ax.set_xticks([])  # Remove tick positions
    ax.set_xticklabels([])  # Remove tick labels

    ax.set_xlabel('')  # Remove x-axis label
    for spine in ax.spines.values():
        spine.set_linewidth(4)

    ax.tick_params(axis='both', which='major', length=12, width=3)
    ax.tick_params(axis='both', which='minor', length=8, width=2)

# Add axis labels and facet titles
g.set_axis_labels("", "", fontsize=40)
g.set_axis_labels("", f"{selected_antibiotic}\nIC$_{{50}}$ (μg/mL)", fontsize=40)
g.set_titles("", size=50)  # Facet title uses the column value

# Adjust overall layout and spacing
g.fig.subplots_adjust(left=0.2, right=0.9, top=0.9, wspace=0.3, hspace=0.4)
g.fig.set_size_inches(15, 10)  # Total figure size

# Save the plot (optional)

g.fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures'/'finals'/'0512_finals'/ f"{selected_antibiotic}_PLPAPC_IC50.png", dpi=600, bbox_inches="tight")

# Show the plot
plt.show()


In [ ]:
import seaborn as sns

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"


########## NO LABELS

# Assign the antibiotic you want to graph
selected_antibiotic = "Cefepime"

# Subset the data for the selected antibiotic
subset_data = growth_features[growth_features['Antibiotic'] == selected_antibiotic].copy()  # Avoid SettingWithCopyWarning

# Custom y-axis limits for each antibiotic
custom_y_limits = {
    "Levofloxacin": (0.001, 10),
    "Amikacin": (.1, 1000),
    "Cefepime": (0.001, 10)
}

# Create a new column for the updated group labels
def assign_group_label(row):
    if row['group'] == "P":
        return r"MG"  # Just MG
    elif row['group'] == "PA":
        return r"MG$^{\mathrm{AMI}}$"  # MG with AMI as superscript
    elif row['group'] == "PC":
        return r"MG$^{\mathrm{CEF}}$"  # MG with CEF as superscript
    elif row['group'] == "PL":
        return r"MG$^{\mathrm{LEV}}$"  # MG with LEV as superscript
    else:
        return "Unknown"

# Ensure the 'group' column exists
if 'group' not in subset_data.columns:
    subset_data['group'] = growth_features['Strain'].apply(lambda x: re.split(r'[1-9]', x)[0])

# Apply the function to create the new column
subset_data.loc[:, 'group_fancy'] = subset_data.apply(assign_group_label, axis=1)

# Update the palette to match the new group labels
group_palette = {
    r"MG": "grey",
    r"MG$^{\mathrm{AMI}}$": "blue",
    r"MG$^{\mathrm{CEF}}$": "forestgreen",
    r"MG$^{\mathrm{LEV}}$": "red",
}
order = [r"MG", r"MG$^{\mathrm{AMI}}$", r"MG$^{\mathrm{LEV}}$", r"MG$^{\mathrm{CEF}}$"]

# Use FacetGrid for plotting
g = sns.FacetGrid(
    subset_data,
    col="Antibiotic",  # Facet by Antibiotic
    sharey=False,  # Allow independent y-axis scaling
    height=10,  # Height of each facet
    aspect=1.5,  # Aspect ratio for width scaling
)

# Map bar plots to the grid
g.map(
    sns.barplot,
    "group_fancy",  # X-axis
    "IC50",  # Y-axis
    ci="sd",
    order=order,  # Ensure consistent x-axis order
    palette=group_palette,
    alpha=0.6,
    errcolor="black",  # Color of the error bars
    errwidth=3,  # Thickness of the error bars
    capsize=0.05  # Length of the caps on error bars
)

# Custom function to calculate evenly spaced x-positions
def get_evenly_spaced_positions(group_data, x_base, spread=0.36, gap=0.08):
    """
    Calculate evenly spaced x-positions around a central x_base, leaving a gap in the middle.

    Parameters:
        group_data: DataFrame containing points for a single category.
        x_base: Central x position for the category.
        spread: Maximum range to spread points around the center.
        gap: The size of the gap to leave in the middle (default: 0.1).

    Returns:
        List of adjusted x-positions for the points.
    """
    n_points = len(group_data)
    if n_points == 1:
        return [x_base]  # No need to spread if there's only one point

    half_points = n_points // 2  # Divide points into two halves
    if n_points % 2 == 0:
        # Even number of points
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        offsets = np.concatenate([left_offsets, right_offsets])
    else:
        # Odd number of points: one point stays at the center
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        center_offset = [0]  # Keep one point at the exact center
        offsets = np.concatenate([left_offsets, center_offset, right_offsets])

    return x_base + offsets

# Overlay strip plots with custom spacing
for group_fancy, group_data in subset_data.groupby('group_fancy'):
    x_base = order.index(group_fancy)  # Base x position based on the custom order
    x_positions = get_evenly_spaced_positions(group_data, x_base)
    
    g.ax.scatter(
        x_positions,
        group_data['IC50'],
        color="black",
        facecolor="black",
        s=100,
        edgecolor="black",
        linewidth=2,
        zorder=10  # Ensure points are above other plot elements
    )

# Customizations for the y-axis and ticks
for ax in g.axes.flat:
    ax.set_yscale("log")  # Set log scale

    # Apply custom y-axis limits for the selected antibiotic first
    if selected_antibiotic in custom_y_limits:
        ax.set_ylim(custom_y_limits[selected_antibiotic])

    # Now adjust font size for all y-tick labels (after y-limits are set)
    for label in ax.get_yticklabels():
        label.set_fontsize(37)

    ax.set_xticks([])  # Remove tick positions
    ax.set_xticklabels([])  # Remove tick labels

    ax.set_xlabel('')  # Remove x-axis label
    for spine in ax.spines.values():
        spine.set_linewidth(4)

    ax.tick_params(axis='both', which='major', length=12, width=3)
    ax.tick_params(axis='both', which='minor', length=8, width=2)

# Add axis labels and facet titles
g.set_axis_labels("", "", fontsize=40)
g.set_axis_labels("", f"{selected_antibiotic}\nIC$_{{50}}$ (μg/mL)", fontsize=40)
g.set_titles("", size=50)  # Facet title uses the column value

# Adjust overall layout and spacing
g.fig.subplots_adjust(left=0.2, right=0.9, top=0.9, wspace=0.3, hspace=0.4)
g.fig.set_size_inches(15, 10)  # Total figure size

# Save the plot (optional)

g.fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures'/'finals'/'0512_finals'/ f"{selected_antibiotic}_PLPAPC_IC50.png", dpi=600, bbox_inches="tight")

# Show the plot
plt.show()


In [ ]:
######### LABELS
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Nimbus Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Nimbus Roman"

# Assign the antibiotic you want to graph
selected_antibiotic = "Cefepime"

# Subset the data for the selected antibiotic
subset_data = growth_features[growth_features['Antibiotic'] == selected_antibiotic].copy()  # Avoid SettingWithCopyWarning

# Custom y-axis limits for each antibiotic
custom_y_limits = {
    "Levofloxacin": (0.001, 10),
    "Amikacin": (.1, 1000),
    "Cefepime": (0.001, 10)
}

# Create a new column for the updated group labels
def assign_group_label(row):
    if row['group'] == "P":
        return r"MG"  # Just MG
    elif row['group'] == "PA":
        return r"MG$^{\mathrm{AMI}}$"  # MG with AMI as superscript
    elif row['group'] == "PC":
        return r"MG$^{\mathrm{CEF}}$"  # MG with CEF as superscript
    elif row['group'] == "PL":
        return r"MG$^{\mathrm{LEV}}$"  # MG with LEV as superscript
    else:
        return "Unknown"

# Ensure the 'group' column exists
if 'group' not in subset_data.columns:
    subset_data['group'] = growth_features['Strain'].apply(lambda x: re.split(r'[1-9]', x)[0])

# Apply the function to create the new column
subset_data.loc[:, 'group_fancy'] = subset_data.apply(assign_group_label, axis=1)

# Update the palette to match the new group labels
group_palette = {
    r"MG": "grey",
    r"MG$^{\mathrm{AMI}}$": "blue",
    r"MG$^{\mathrm{CEF}}$": "forestgreen",
    r"MG$^{\mathrm{LEV}}$": "red",
}
order = [r"MG", r"MG$^{\mathrm{AMI}}$", r"MG$^{\mathrm{LEV}}$", r"MG$^{\mathrm{CEF}}$"]

# Use FacetGrid for plotting
g = sns.FacetGrid(
    subset_data,
    col="Antibiotic",  # Facet by Antibiotic
    sharey=False,  # Allow independent y-axis scaling
    height=10,  # Height of each facet
    aspect=1.5,  # Aspect ratio for width scaling
)

# Map bar plots to the grid
g.map(
    sns.barplot,
    "group_fancy",  # X-axis
    "IC50",  # Y-axis
    ci="sd",
    order=order,  # Ensure consistent x-axis order
    palette=group_palette,
    alpha=0.6,
    errcolor="black",  # Color of the error bars
    errwidth=3,  # Thickness of the error bars
    capsize=0.05  # Length of the caps on error bars
)

# Custom function to calculate evenly spaced x-positions
def get_evenly_spaced_positions(group_data, x_base, spread=0.36, gap=0.08):
    """
    Calculate evenly spaced x-positions around a central x_base, leaving a gap in the middle.

    Parameters:
        group_data: DataFrame containing points for a single category.
        x_base: Central x position for the category.
        spread: Maximum range to spread points around the center.
        gap: The size of the gap to leave in the middle (default: 0.1).

    Returns:
        List of adjusted x-positions for the points.
    """
    n_points = len(group_data)
    if n_points == 1:
        return [x_base]  # No need to spread if there's only one point

    half_points = n_points // 2  # Divide points into two halves
    if n_points % 2 == 0:
        # Even number of points
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        offsets = np.concatenate([left_offsets, right_offsets])
    else:
        # Odd number of points: one point stays at the center
        left_offsets = np.linspace(-spread, -gap, half_points)
        right_offsets = np.linspace(gap, spread, half_points)
        center_offset = [0]  # Keep one point at the exact center
        offsets = np.concatenate([left_offsets, center_offset, right_offsets])

    return x_base + offsets

# Overlay strip plots with custom spacing
for group_fancy, group_data in subset_data.groupby('group_fancy'):
    x_base = order.index(group_fancy)  # Base x position based on the custom order
    x_positions = get_evenly_spaced_positions(group_data, x_base)
    
    g.ax.scatter(
        x_positions,
        group_data['IC50'],
        color="black",
        facecolor="black",
        s=100,
        edgecolor="black",
        linewidth=2,
        zorder=10  # Ensure points are above other plot elements
    )


# Customizations for the y-axis and ticks
for ax in g.axes.flat:
    ax.set_yscale("log")  # Set log scale

    # Apply custom y-axis limits for the selected antibiotic first
    if selected_antibiotic in custom_y_limits:
        ax.set_ylim(custom_y_limits[selected_antibiotic])

    # Now adjust font size for all y-tick labels (after y-limits are set)
    for label in ax.get_yticklabels():
        label.set_fontsize(37)

 # Set x-tick positions and labels explicitly
    tick_positions = range(len(subset_data['group_fancy'].unique()))  # Create positions for each category
    ax.set_xticks(tick_positions)  # Set tick positions
    ax.set_xticklabels(subset_data['group_fancy'].unique(), rotation=45, ha="center", fontsize=37) 
    
    # Remove x-axis label

    for spine in ax.spines.values():
        spine.set_linewidth(4)  # Set thickness of the spine lines

    ax.tick_params(axis='both', which='major', length=12, width=3)  # Major ticks
    ax.tick_params(axis='both', which='minor', length=8, width=2)  # Minor ticks

# Add axis labels and facet titles
# g.set_axis_labels("Cohort", "", fontsize=40)
g.set_axis_labels("", f"{selected_antibiotic}\nIC$_{{50}}$ (μg/mL)", fontsize=40)
g.set_titles("", size=50)  # Facet title uses the column value

# Adjust overall layout and spacing
g.fig.subplots_adjust(left=0.2, right=0.9, top=0.9, wspace=0.3, hspace=0.4)
g.fig.set_size_inches(15, 10)  # Total figure size

# Save the plot (optional)
g.fig.savefig(PROJECT_PATH / PROJECT_ID / 'figures'/'finals'/'0512_finals'/ f"{selected_antibiotic}_PLPCPA_IC50_labels.png", dpi=600, bbox_inches="tight")

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import itertools
import re
from scipy.stats import ttest_rel

# Assuming growth_features DataFrame is already defined
# Step 1: Create the 'group' and 'culturenumber' columns
growth_features['group'] = growth_features['Strain'].apply(lambda x: re.split(r'[1-9]', x)[0])
growth_features['culturenumber'] = growth_features['Strain'].apply(lambda x: re.search(r'(\d+)', x).group() if re.search(r'(\d+)', x) else None)

# Step 2: Initialize an empty list to store results and get all unique pairs of groups
unique_groups = growth_features['group'].unique()
group_pairs = list(itertools.combinations(unique_groups, 2))  # Generate all unique pairs of groups

# Step 3: Perform paired t-tests for each unique pair of groups within each antibiotic
results = []
for antibiotic in growth_features['Antibiotic'].unique():
    # Filter data by antibiotic
    antibiotic_data = growth_features[growth_features['Antibiotic'] == antibiotic]
    
    for group1, group2 in group_pairs:
        # Filter data for each group
        data1 = antibiotic_data[antibiotic_data['group'] == group1]
        data2 = antibiotic_data[antibiotic_data['group'] == group2]
        
        # Find common culture numbers by using set intersection
        common_culturenumbers = set(data1['culturenumber']).intersection(set(data2['culturenumber']))
        data1_common = data1[data1['culturenumber'].isin(common_culturenumbers)].set_index('culturenumber')
        data2_common = data2[data2['culturenumber'].isin(common_culturenumbers)].set_index('culturenumber')
        
        # Ensure paired data has matching indices (culturenumber)
        if not data1_common.empty and not data2_common.empty:
            ic50_group1 = data1_common['IC50']
            ic50_group2 = data2_common['IC50']
            
            # Perform paired t-test
            t_stat, p_value = ttest_rel(ic50_group1, ic50_group2)
            
            # Append results to list
            results.append({
                'Antibiotic': antibiotic,
                'Group1': group1,
                'Group2': group2,
                't_stat': t_stat,
                'p_value': p_value
            })

# Step 4: Convert results into a DataFrame
sig_growth_features = pd.DataFrame(results)
sig_growth_features_filtered = sig_growth_features.query("Group1 == 'P'")

# Display or save the results
sig_growth_features_filtered  # Print to check if results are populated


In [ ]:
import pandas as pd

# Step 1: Calculate the mean IC50 for each combination of Antibiotic and group
group_means = growth_features.groupby(['Antibiotic', 'group'])['IC50'].mean().reset_index()
group_means = group_means.rename(columns={'IC50': 'mean_IC50'})

# Step 2: Merge the means for Group1 and Group2 into sig_growth_features
sig_growth_features = sig_growth_features.merge(
    group_means, left_on=['Antibiotic', 'Group1'], right_on=['Antibiotic', 'group'], how='left'
).rename(columns={'mean_IC50': 'mean_IC50_Group1'}).drop(columns='group')

sig_growth_features = sig_growth_features.merge(
    group_means, left_on=['Antibiotic', 'Group2'], right_on=['Antibiotic', 'group'], how='left'
).rename(columns={'mean_IC50': 'mean_IC50_Group2'}).drop(columns='group')

# Step 3: Filter for rows where Group1 is "P"
sig_growth_features_filtered = sig_growth_features[sig_growth_features['Group1'] == 'P']

# Display the filtered DataFrame
sig_growth_features_filtered
